# IDEA #5: PRODUCT CATEGORY PERFORMANCE ANALYSIS
## Category Revenue, Profitability, Growth Trend Deep-dive

---

**Objective**: Analyze business performance by product category.
- Identify top/bottom performing categories
- Understand category profitability and margin disparities
- Forecast category revenue trends
- Recommend product strategy, inventory mix, and category expansion priorities

---

In [1]:
# Hiển thêm: install dependencies cho notebook (chạy 1 lần đầu — sau đó comment lại để tránh re-install)
%pip install -q numpy pandas matplotlib seaborn scipy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# Config
np.random.seed(42)
print(f'✓ Pandas: {pd.__version__}')
print(f'✓ NumPy: {np.__version__}')
print(f'✓ Seed: 42')

# Path discovery
cwd = Path.cwd()
for i in range(5):
    if (cwd / 'data' / 'datathon-2026-round-1').exists():
        DATA_DIR = cwd / 'data' / 'datathon-2026-round-1'
        break
    cwd = cwd.parent

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'✓ Data path: {DATA_DIR}')
print(f'✓ Output path: {OUTPUT_DIR.resolve()}')

✓ Pandas: 3.0.2
✓ NumPy: 2.4.4
✓ Seed: 42
✓ Data path: /Users/dominhhien/Documents/AI/Datathon/TuNgayToiGapEm/data/datathon-2026-round-1
✓ Output path: /Users/dominhhien/Documents/AI/Datathon/TuNgayToiGapEm/phan_2_eda/outputs/idea_5/outputs


## CELL 2: LOAD DATA

In [3]:
print('='*70)
print('LOADING DATA')
print('='*70)

products = pd.read_csv(DATA_DIR / 'products.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
returns = pd.read_csv(DATA_DIR / 'returns.csv')

print(f'✓ Products shape: {products.shape}')
print(f'✓ Order items shape: {order_items.shape}')
print(f'✓ Orders shape: {orders.shape}')
print(f'✓ Returns shape: {returns.shape}')

LOADING DATA
✓ Products shape: (2412, 8)
✓ Order items shape: (714669, 7)
✓ Orders shape: (646945, 8)
✓ Returns shape: (39939, 7)


## CELL 3: BUILD CATEGORY FACT TABLE

In [4]:
# Merge order items with products
order_items_cat = order_items.merge(products[['product_id', 'category', 'cogs']], 
                                     on='product_id', how='left')

# Calculate revenue and COGS
order_items_cat['revenue'] = order_items_cat['quantity'] * order_items_cat['unit_price'] - order_items_cat['discount_amount']
order_items_cat['cogs'] = order_items_cat['quantity'] * order_items_cat['cogs']
order_items_cat['gross_profit'] = order_items_cat['revenue'] - order_items_cat['cogs']
order_items_cat['margin_pct'] = (order_items_cat['gross_profit'] / order_items_cat['revenue'] * 100).fillna(0)

# Category revenue table
category_revenue = order_items_cat.groupby('category').agg({
    'order_id': 'nunique',
    'quantity': 'sum',
    'revenue': 'sum',
    'cogs': 'sum',
    'gross_profit': 'sum',
    'product_id': 'nunique'
}).reset_index()
category_revenue.columns = ['category', 'orders', 'quantity', 'revenue', 'cogs', 'gross_profit', 'sku_count']
category_revenue['aov'] = category_revenue['revenue'] / category_revenue['orders']
category_revenue['margin_pct'] = (category_revenue['gross_profit'] / category_revenue['revenue'] * 100)

# Category return rate (returns have order_id, product_id directly)
returns_cat = returns.merge(products[['product_id', 'category']], 
                              on='product_id', how='left')

category_return_rate = returns_cat.groupby('category').agg({
    'return_id': 'count',
    'refund_amount': 'sum'
}).reset_index()
category_return_rate.columns = ['category', 'return_lines', 'refund_amount']

# Merge for complete picture
category_fact = category_revenue.merge(category_return_rate, on='category', how='left')
category_fact['return_lines'] = category_fact['return_lines'].fillna(0)
category_fact['refund_amount'] = category_fact['refund_amount'].fillna(0)
category_fact['return_rate'] = category_fact['return_lines'] / category_fact['quantity']

print(f'✓ Category fact table shape: {category_fact.shape}')
print(f'✓ Categories: {category_fact["category"].nunique()}')

✓ Category fact table shape: (4, 12)
✓ Categories: 4


## CELL 4: DESCRIPTIVE — CATEGORY OVERVIEW

In [5]:
print('='*70)
print('TẦNG 1: DESCRIPTIVE — CATEGORY OVERVIEW')
print('='*70)

# Total metrics
total_revenue = category_fact['revenue'].sum()
total_orders = category_fact['orders'].sum()
total_aov = total_revenue / total_orders
total_gross_profit = category_fact['gross_profit'].sum()
total_margin_pct = (total_gross_profit / total_revenue * 100)
total_return_rate = category_fact['return_lines'].sum() / category_fact['quantity'].sum()

print(f'\n📊 Overall metrics:')
print(f'  - Total revenue: ${total_revenue:,.0f}')
print(f'  - Total gross profit: ${total_gross_profit:,.0f}')
print(f'  - Overall margin: {total_margin_pct:.2f}%')
print(f'  - Total orders: {total_orders:,.0f}')
print(f'  - Overall AOV: ${total_aov:,.2f}')
print(f'  - Total return rate: {total_return_rate*100:.2f}%')

# By category
category_sorted = category_fact.sort_values('revenue', ascending=False)
category_sorted['revenue_share'] = category_sorted['revenue'] / category_sorted['revenue'].sum() * 100

print('\n📍 Top categories by revenue:')
for idx, (_, row) in enumerate(category_sorted.head(10).iterrows(), 1):
    print(f"{idx:2}. {row['category']:15} | ${row['revenue']:>12,.0f} | {row['margin_pct']:>6.2f}% margin | "
          f"{row['return_rate']*100:>5.2f}% return | {row['revenue_share']:>5.1f}% share")

TẦNG 1: DESCRIPTIVE — CATEGORY OVERVIEW

📊 Overall metrics:
  - Total revenue: $15,680,869,265
  - Total gross profit: $1,517,418,746
  - Overall margin: 9.68%
  - Total orders: 650,651
  - Overall AOV: $24,100.28
  - Total return rate: 1.24%

📍 Top categories by revenue:
 1. Streetwear      | $12,558,477,099 |   9.28% margin |  1.23% return |  80.1% share
 2. Outdoor         | $2,353,396,797 |  11.35% margin |  1.26% return |  15.0% share
 3. Casual          | $ 440,285,194 |   7.66% margin |  1.20% return |   2.8% share
 4. GenZ            | $ 328,710,176 |  15.47% margin |  1.27% return |   2.1% share


## CELL 5: DIAGNOSTIC — PROFITABILITY ANALYSIS

In [6]:
print('='*70)
print('TẦNG 2: DIAGNOSTIC — CATEGORY PROFITABILITY')
print('='*70)

# Top/bottom by profit
top_profit = category_fact.nlargest(3, 'gross_profit')
bottom_profit = category_fact.nsmallest(3, 'gross_profit')

print('\n🏆 Top 3 categories by gross profit:')
for _, row in top_profit.iterrows():
    print(f"  {row['category']:15} | ${row['gross_profit']:>12,.0f} | {row['margin_pct']:>6.2f}% margin")

print('\n📉 Bottom 3 categories by gross profit:')
for _, row in bottom_profit.iterrows():
    print(f"  {row['category']:15} | ${row['gross_profit']:>12,.0f} | {row['margin_pct']:>6.2f}% margin")

# Margin analysis
highest_margin = category_fact.loc[category_fact['margin_pct'].idxmax()]
lowest_margin = category_fact.loc[category_fact['margin_pct'].idxmin()]

print('\n💰 Margin comparison:')
print(f"  Highest: {highest_margin['category']:15} | {highest_margin['margin_pct']:>6.2f}%")
print(f"  Lowest:  {lowest_margin['category']:15} | {lowest_margin['margin_pct']:>6.2f}%")
print(f"  Spread:  {(highest_margin['margin_pct'] - lowest_margin['margin_pct']):>6.2f} pp")

# Calculate revenue share
total_revenue = category_fact['revenue'].sum()
category_fact['revenue_share'] = category_fact['revenue'] / total_revenue

# Efficiency score
category_fact['efficiency_score'] = (category_fact['margin_pct'] * category_fact['revenue_share']) - (category_fact['return_rate'] * 100)
best_efficiency = category_fact.loc[category_fact['efficiency_score'].idxmax()]
worst_efficiency = category_fact.loc[category_fact['efficiency_score'].idxmin()]

print('\n🎯 Efficiency ranking (margin × revenue_share - return_rate):')
print(f"  Best:  {best_efficiency['category']:15} | Score: {best_efficiency['efficiency_score']:>6.2f}")
print(f"  Worst: {worst_efficiency['category']:15} | Score: {worst_efficiency['efficiency_score']:>6.2f}")

TẦNG 2: DIAGNOSTIC — CATEGORY PROFITABILITY

🏆 Top 3 categories by gross profit:
  Streetwear      | $1,165,807,512 |   9.28% margin
  Outdoor         | $ 267,034,092 |  11.35% margin
  GenZ            | $  50,836,377 |  15.47% margin

📉 Bottom 3 categories by gross profit:
  Casual          | $  33,740,765 |   7.66% margin
  GenZ            | $  50,836,377 |  15.47% margin
  Outdoor         | $ 267,034,092 |  11.35% margin

💰 Margin comparison:
  Highest: GenZ            |  15.47%
  Lowest:  Casual          |   7.66%
  Spread:    7.80 pp

🎯 Efficiency ranking (margin × revenue_share - return_rate):
  Best:  Streetwear      | Score:   6.20
  Worst: Casual          | Score:  -0.99


## CELL 5b: Streetwear Concentration Sensitivity Analysis

**Hiển thêm**: Streetwear chiếm 80.1% revenue → concentration risk. Tính sensitivity của blended margin & gross profit khi Streetwear demand shock ±10%, ±20%, ±30%. Bổ sung Welch's t-test cho margin gap giữa Streetwear vs các category khác để confirm khác biệt có ý nghĩa thống kê.

**Phương pháp**:
- Sensitivity: `blended_margin_shock(δ) = Σ (revenue_cat × (1 + δ_cat)) × margin_cat / Σ (revenue_cat × (1 + δ_cat))` với `δ_streetwear ∈ {-30%, -20%, -10%, 0, +10%, +20%, +30%}`, các category khác giữ nguyên.
- **Welch's t-test** trên `margin_pct` per-order-line: H₀ Streetwear vs others có mean margin bằng nhau. Welch dùng vì variance không equal.

In [7]:
# ============================================================================
# CELL 5c: STREETWEAR CONCENTRATION SENSITIVITY + WELCH'S T-TEST
# Hiển thêm: sensitivity analysis + statistical test (T2 narrative supporting)
# ============================================================================
from scipy.stats import ttest_ind

print('='*70)
print('STREETWEAR CONCENTRATION RISK — SENSITIVITY ANALYSIS')
print('='*70)

# 1) Welch's t-test: margin Streetwear vs các category khác (per-order-line level)
sw_mask = order_items_cat['category'] == 'Streetwear'
margin_sw     = order_items_cat.loc[sw_mask, 'margin_pct'].dropna().values
margin_other  = order_items_cat.loc[~sw_mask, 'margin_pct'].dropna().values

t_stat, t_p = ttest_ind(margin_sw, margin_other, equal_var=False)  # Welch
print(f'\n📊 Welch t-test (margin Streetwear vs others, per-order-line):')
print(f'  n_streetwear = {len(margin_sw):,} | mean = {margin_sw.mean():.2f}%')
print(f'  n_others     = {len(margin_other):,} | mean = {margin_other.mean():.2f}%')
print(f'  t = {t_stat:.3f} | p = {t_p:.6e}')
verdict_t = 'REJECT H₀ — margin gap significant' if t_p < 0.05 else 'FAIL TO REJECT — no significant gap'
print(f'  Verdict @ α=0.05: {verdict_t}')

# 2) Sensitivity: blended margin & gross profit khi Streetwear demand shock ±X%
shocks = [-0.30, -0.20, -0.10, 0.0, 0.10, 0.20, 0.30]
base_total_rev = category_fact['revenue'].sum()
base_blended_margin = (category_fact['revenue'] * category_fact['margin_pct']).sum() / base_total_rev

rows = []
for delta in shocks:
    cat_shock = category_fact.copy()
    cat_shock.loc[cat_shock['category'] == 'Streetwear', 'revenue'] *= (1 + delta)
    new_total_rev = cat_shock['revenue'].sum()
    new_gp = (cat_shock['revenue'] * cat_shock['margin_pct'] / 100).sum()
    new_margin = new_gp / new_total_rev * 100
    delta_margin_pp = new_margin - base_blended_margin
    delta_gp = new_gp - (base_blended_margin / 100) * base_total_rev
    rows.append({
        'shock_streetwear_pct': delta * 100,
        'new_total_revenue':    new_total_rev,
        'new_blended_margin':   new_margin,
        'delta_margin_pp':      delta_margin_pp,
        'delta_gross_profit':   delta_gp,
    })
sensitivity = pd.DataFrame(rows)
print(f'\n📊 Sensitivity table (Streetwear demand shock → blended margin & GP):')
print(sensitivity.round(3).to_string(index=False))

# Rule of thumb derived from sensitivity slope
m_per_10pct = (sensitivity.loc[sensitivity['shock_streetwear_pct'] == -10, 'delta_margin_pp'].iloc[0])
gp_per_10pct = (sensitivity.loc[sensitivity['shock_streetwear_pct'] == -10, 'delta_gross_profit'].iloc[0])
print(f'\n🎯 Rule of thumb:')
print(f'  Streetwear demand shock −10% → blended margin Δ {m_per_10pct:+.3f}pp; '
      f'GP Δ ${gp_per_10pct:,.0f} ({gp_per_10pct/1e6:+.2f}M)')
print(f'  → Concentration risk: 80.1% revenue dồn 1 category → diversify Trendy/Activewear (idea 11) là hedging.')

# Save sensitivity table cho báo cáo PDF
sensitivity.round(3).to_csv(OUTPUT_DIR / 'streetwear_sensitivity.csv', index=False)
print(f'\n✓ Saved: streetwear_sensitivity.csv')


STREETWEAR CONCENTRATION RISK — SENSITIVITY ANALYSIS

📊 Welch t-test (margin Streetwear vs others, per-order-line):
  n_streetwear = 393,533 | mean = 5.89%
  n_others     = 321,136 | mean = 8.55%
  t = -42.469 | p = 0.000000e+00
  Verdict @ α=0.05: REJECT H₀ — margin gap significant

📊 Sensitivity table (Streetwear demand shock → blended margin & GP):
 shock_streetwear_pct  new_total_revenue  new_blended_margin  delta_margin_pp  delta_gross_profit
                -30.0       1.191333e+10               9.801            0.125       -3.497423e+08
                -20.0       1.316917e+10               9.752            0.075       -2.331615e+08
                -10.0       1.442502e+10               9.711            0.034       -1.165808e+08
                  0.0       1.568087e+10               9.677            0.000        0.000000e+00
                 10.0       1.693672e+10               9.648           -0.029        1.165808e+08
                 20.0       1.819256e+10               9.6

## CELL 6: PREDICTIVE — CATEGORY REVENUE FORECAST

In [8]:
# Build monthly revenue by category
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['year_month'] = orders['order_date'].dt.to_period('M')

monthly_cat = order_items_cat.copy()
monthly_cat['order_date'] = pd.to_datetime(orders.set_index('order_id').loc[monthly_cat['order_id'], 'order_date'].values)
monthly_cat['year_month'] = monthly_cat['order_date'].dt.to_period('M')

monthly_cat_revenue = monthly_cat.groupby(['year_month', 'category'])['revenue'].sum().reset_index()

# Top category trend + forecast
top_cat_name = category_fact.loc[category_fact['revenue'].idxmax(), 'category']
top_cat_data = monthly_cat_revenue[monthly_cat_revenue['category'] == top_cat_name].copy()
top_cat_data = top_cat_data.sort_values('year_month')

if len(top_cat_data) > 3:
    X = np.arange(len(top_cat_data)).reshape(-1, 1)
    y = top_cat_data['revenue'].values
    model = LinearRegression().fit(X, y)
    
    # Forecast next 3 months
    future_X = np.array([[len(top_cat_data)], [len(top_cat_data)+1], [len(top_cat_data)+2]])
    forecast_revenue = model.predict(future_X)
    current_revenue = top_cat_data['revenue'].iloc[-1]
    trend_slope = model.coef_[0]
    
    print('='*70)
    print('TẦNG 3: PREDICTIVE — CATEGORY GROWTH FORECAST')
    print('='*70)
    print(f'\n🔮 Top category ({top_cat_name}) forecast:')
    print(f'  Current month revenue: ${current_revenue:,.0f}')
    print(f'  Forecast month +1: ${forecast_revenue[0]:,.0f}')
    print(f'  Forecast month +2: ${forecast_revenue[1]:,.0f}')
    print(f'  Forecast month +3: ${forecast_revenue[2]:,.0f}')
    print(f'  Trend slope: ${trend_slope:,.0f}/month')
else:
    forecast_revenue = [0, 0, 0]
    trend_slope = 0
    print('⚠ Insufficient data for forecast')

TẦNG 3: PREDICTIVE — CATEGORY GROWTH FORECAST

🔮 Top category (Streetwear) forecast:
  Current month revenue: $31,135,925
  Forecast month +1: $71,725,039
  Forecast month +2: $71,284,954
  Forecast month +3: $70,844,869
  Trend slope: $-440,085/month


## CELL 7: CHART 1 — CATEGORY REVENUE RANKING

In [9]:
fig, ax = plt.subplots(figsize=(12, 8))
cat_sorted_chart = category_sorted.sort_values('revenue', ascending=True).tail(15)
colors = ['#2ecc71' if x == cat_sorted_chart['revenue'].max() else '#3498db' if x > cat_sorted_chart['revenue'].median() else '#e74c3c' 
          for x in cat_sorted_chart['revenue']]
ax.barh(cat_sorted_chart['category'], cat_sorted_chart['revenue'], color=colors)
ax.set_xlabel('Revenue ($)', fontsize=11, fontweight='bold')
ax.set_title('Top 15 Categories by Revenue', fontsize=13, fontweight='bold')
for i, v in enumerate(cat_sorted_chart['revenue']):
    ax.text(v, i, f' ${v/1e9:.2f}B', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_category_revenue_ranking.png', dpi=100, bbox_inches='tight')
plt.close()
print('✓ Saved: 01_category_revenue_ranking.png')

✓ Saved: 01_category_revenue_ranking.png


## CELL 8: CHART 2 — MARGIN vs REVENUE SCATTER

In [10]:
fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(category_fact['margin_pct'], category_fact['revenue'], 
                     s=category_fact['return_rate']*1000+50, alpha=0.6, c=category_fact['gross_profit'], cmap='viridis')
for idx, row in category_fact.iterrows():
    ax.annotate(row['category'], (row['margin_pct'], row['revenue']), fontsize=8, ha='center')
ax.set_xlabel('Gross Margin (%)', fontsize=11, fontweight='bold')
ax.set_ylabel('Revenue ($)', fontsize=11, fontweight='bold')
ax.set_title('Category Margin vs Revenue (bubble size = return rate)', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=ax, label='Gross Profit ($)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_margin_vs_revenue_scatter.png', dpi=100, bbox_inches='tight')
plt.close()
print('✓ Saved: 02_margin_vs_revenue_scatter.png')

✓ Saved: 02_margin_vs_revenue_scatter.png


## CELL 9: CHART 3 — TOP/BOTTOM CATEGORY COMPARISON

In [11]:
comparison_cats = pd.concat([category_fact.nlargest(5, 'revenue'), 
                             category_fact.nsmallest(5, 'revenue')])
comparison_cats['group'] = comparison_cats['revenue'].apply(
    lambda x: 'Top 5' if x >= category_fact['revenue'].nlargest(5).min() else 'Bottom 5'
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Revenue
for group, color in [('Top 5', '#2ecc71'), ('Bottom 5', '#e74c3c')]:
    data = comparison_cats[comparison_cats['group'] == group]
    ax1.bar(data['category'], data['revenue'], label=group, alpha=0.8, color=color)
ax1.set_ylabel('Revenue ($)', fontsize=11, fontweight='bold')
ax1.set_title('Top 5 vs Bottom 5 Categories - Revenue', fontsize=12, fontweight='bold')
ax1.legend()
ax1.tick_params(axis='x', rotation=45)

# Margin
for group, color in [('Top 5', '#2ecc71'), ('Bottom 5', '#e74c3c')]:
    data = comparison_cats[comparison_cats['group'] == group]
    ax2.bar(data['category'], data['margin_pct'], label=group, alpha=0.8, color=color)
ax2.set_ylabel('Gross Margin (%)', fontsize=11, fontweight='bold')
ax2.set_title('Top 5 vs Bottom 5 Categories - Margin', fontsize=12, fontweight='bold')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_top_bottom_categories_comparison.png', dpi=100, bbox_inches='tight')
plt.close()
print('✓ Saved: 03_top_bottom_categories_comparison.png')

✓ Saved: 03_top_bottom_categories_comparison.png


## CELL 10: CHART 4 — TOP CATEGORY MONTHLY TREND

In [12]:
fig, ax = plt.subplots(figsize=(13, 5))
top_cat_data_sorted = top_cat_data.sort_values('year_month')
ax.plot(range(len(top_cat_data_sorted)), top_cat_data_sorted['revenue'], 
        marker='o', linewidth=2, markersize=6, label='Actual', color='#3498db')

# Add trend line
if len(top_cat_data_sorted) > 3:
    X_all = np.arange(len(top_cat_data_sorted)).reshape(-1, 1)
    trend_line = model.predict(X_all)
    ax.plot(range(len(top_cat_data_sorted)), trend_line, 
            '--', linewidth=2, label='Trend', color='#e74c3c', alpha=0.7)
    
    # Forecast
    forecast_x = [len(top_cat_data_sorted)-1, len(top_cat_data_sorted), len(top_cat_data_sorted)+1, len(top_cat_data_sorted)+2]
    forecast_y = [top_cat_data_sorted['revenue'].iloc[-1]] + list(forecast_revenue)
    ax.plot(forecast_x, forecast_y, ':', linewidth=2, marker='s', markersize=5, 
            label='Forecast', color='#2ecc71', alpha=0.7)

ax.set_xlabel('Month', fontsize=11, fontweight='bold')
ax.set_ylabel('Revenue ($)', fontsize=11, fontweight='bold')
ax.set_title(f'{top_cat_name} Category - Monthly Revenue Trend + Forecast', fontsize=13, fontweight='bold')
ax.legend(loc='best')
plt.xticks(range(0, len(top_cat_data_sorted), max(1, len(top_cat_data_sorted)//6)))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_top_category_trend_forecast.png', dpi=100, bbox_inches='tight')
plt.close()
print('✓ Saved: 04_top_category_trend_forecast.png')

✓ Saved: 04_top_category_trend_forecast.png


## CELL 11: PRESCRIPTIVE — CATEGORY STRATEGY

In [13]:
print('='*70)
print('TẦNG 4: PRESCRIPTIVE — CATEGORY OPTIMIZATION STRATEGY')
print('='*70)

# Calculate efficiency score for ranking
category_fact_sorted = category_fact.sort_values('efficiency_score', ascending=False)

print('\n📊 RECOMMENDATION 1: CATEGORY PRIORITIZATION')
print(f'  - Growth categories (high revenue + high margin):')
growth_cats = category_fact[(category_fact['revenue_share'] > 5) & (category_fact['margin_pct'] > total_margin_pct)]
for _, row in growth_cats.iterrows():
    print(f'    • {row["category"]}: ${row["revenue"]:,.0f} | {row["margin_pct"]:.2f}% margin')

print(f'\n  - Cash cow categories (high revenue but lower margin):')
cash_cows = category_fact[(category_fact['revenue_share'] > 5) & (category_fact['margin_pct'] <= total_margin_pct)]
for _, row in cash_cows.iterrows():
    print(f'    • {row["category"]}: ${row["revenue"]:,.0f} | {row["margin_pct"]:.2f}% margin')

print('\n📊 RECOMMENDATION 2: MARGIN OPTIMIZATION')
margin_gap_target = highest_margin['margin_pct'] - lowest_margin['margin_pct']
print(f'  - Margin spread: {margin_gap_target:.2f} pp (highest vs lowest)')
for _, row in lowest_margin.to_frame().T.iterrows():
    potential_uplift = (row['revenue'] * (highest_margin['margin_pct'] - row['margin_pct']) / 100)
    print(f'  - {row["category"]} uplift potential: ${potential_uplift:,.0f} if reach highest margin')

print('\n📊 RECOMMENDATION 3: INVENTORY MIX OPTIMIZATION')
print(f'  - High-margin fast-movers: increase allocation')
print(f'  - Low-margin high-volume: optimize for velocity')
print(f'  - High-return categories: review quality or position')

print('\n📊 RECOMMENDATION 4: CATEGORY SCORECARD')
print(f'  KPI: revenue_growth, margin%, return_rate, efficiency_score')
print(f'  Frequency: monthly review, quarterly strategy adjustment')

TẦNG 4: PRESCRIPTIVE — CATEGORY OPTIMIZATION STRATEGY

📊 RECOMMENDATION 1: CATEGORY PRIORITIZATION
  - Growth categories (high revenue + high margin):

  - Cash cow categories (high revenue but lower margin):

📊 RECOMMENDATION 2: MARGIN OPTIMIZATION
  - Margin spread: 7.80 pp (highest vs lowest)
  - Casual uplift potential: $34,351,147 if reach highest margin

📊 RECOMMENDATION 3: INVENTORY MIX OPTIMIZATION
  - High-margin fast-movers: increase allocation
  - Low-margin high-volume: optimize for velocity
  - High-return categories: review quality or position

📊 RECOMMENDATION 4: CATEGORY SCORECARD
  KPI: revenue_growth, margin%, return_rate, efficiency_score
  Frequency: monthly review, quarterly strategy adjustment


## CELL 12: EXPORT SUMMARY METRICS

In [14]:
summary_metrics = pd.DataFrame([
    ['Total Revenue', f'${total_revenue:,.0f}'],
    ['Total Gross Profit', f'${total_gross_profit:,.0f}'],
    ['Overall Margin (%)', f'{total_margin_pct:.2f}%'],
    ['Total Orders', f'{total_orders:,.0f}'],
    ['Overall AOV', f'${total_aov:,.2f}'],
    ['Total Return Rate (%)', f'{total_return_rate*100:.2f}%'],
    ['Number of Categories', f'{category_fact.shape[0]}'],
    ['Top Category', top_cat_name],
    ['Top Category Revenue', f'${category_fact.loc[category_fact["category"] == top_cat_name, "revenue"].values[0]:,.0f}'],
    ['Top Category Margin (%)', f'{category_fact.loc[category_fact["category"] == top_cat_name, "margin_pct"].values[0]:.2f}%'],
    ['Highest Margin Category', highest_margin['category']],
    ['Highest Margin (%)', f'{highest_margin["margin_pct"]:.2f}%'],
    ['Lowest Margin Category', lowest_margin['category']],
    ['Lowest Margin (%)', f'{lowest_margin["margin_pct"]:.2f}%'],
    ['Margin Variance (pp)', f'{margin_gap_target:.2f}'],
    ['Top Category Forecast (+1M)', f'${forecast_revenue[0]:,.0f}'],
    ['Top Category Trend Slope ($/month)', f'${trend_slope:,.0f}'],
    ['Best Efficiency Category', best_efficiency['category']],
    ['Worst Efficiency Category', worst_efficiency['category']],
    ['Total Refund Amount', f'${category_fact["refund_amount"].sum():,.0f}']
], columns=['Metric', 'Value'])

summary_metrics.to_csv(OUTPUT_DIR / 'summary_metrics.csv', index=False)
print('\n📊 SUMMARY METRICS TABLE')
print('='*70)
print(summary_metrics.to_string(index=False))
print(f'\n✓ Exported to: {(OUTPUT_DIR / "summary_metrics.csv").resolve()}')


📊 SUMMARY METRICS TABLE
                            Metric           Value
                     Total Revenue $15,680,869,265
                Total Gross Profit  $1,517,418,746
                Overall Margin (%)           9.68%
                      Total Orders         650,651
                       Overall AOV      $24,100.28
             Total Return Rate (%)           1.24%
              Number of Categories               4
                      Top Category      Streetwear
              Top Category Revenue $12,558,477,099
           Top Category Margin (%)           9.28%
           Highest Margin Category            GenZ
                Highest Margin (%)          15.47%
            Lowest Margin Category          Casual
                 Lowest Margin (%)           7.66%
              Margin Variance (pp)            7.80
       Top Category Forecast (+1M)     $71,725,039
Top Category Trend Slope ($/month)       $-440,085
          Best Efficiency Category      Streetwear
      

## CELL 13: FINAL SUMMARY

In [15]:
print('\n' + '='*70)
print('✅ ANALYSIS COMPLETE - IDEA #5 PRODUCT CATEGORY PERFORMANCE')
print('='*70)

print('\n🎯 ONE-LINER INSIGHT:')
print('"Category performance = revenue scale + margin optimization, con duong den toi uu profit."')

print('\n📁 OUTPUT FILES GENERATED:')
print('  ✓ 01_category_revenue_ranking.png - Top 15 categories by revenue')
print('  ✓ 02_margin_vs_revenue_scatter.png - Category efficiency matrix')
print('  ✓ 03_top_bottom_categories_comparison.png - Comparative analysis')
print('  ✓ 04_top_category_trend_forecast.png - Revenue trend & forecast')
print('  ✓ summary_metrics.csv - Key category KPIs')
print(f'\nLocation: {OUTPUT_DIR.resolve()}')

print('\n🔑 KEY FINDINGS:')
print(f'  - Top category: {top_cat_name} ({category_fact.loc[category_fact["category"] == top_cat_name, "revenue_share"].values[0]:.1f}% revenue)')
print(f'  - Margin spread: {margin_gap_target:.2f} pp opportunity')
print(f'  - Overall margin: {total_margin_pct:.2f}% (benchmark for category strategy)')
print(f'  - Forecast: top category trending at ${trend_slope:,.0f}/month')

print('\n🚀 READY FOR:')
print('  ✓ Category expansion roadmap')
print('  ✓ Margin optimization initiatives')
print('  ✓ Inventory mix tuning')
print('  ✓ Pricing & promotional strategy by category')


✅ ANALYSIS COMPLETE - IDEA #5 PRODUCT CATEGORY PERFORMANCE

🎯 ONE-LINER INSIGHT:
"Category performance = revenue scale + margin optimization, con duong den toi uu profit."

📁 OUTPUT FILES GENERATED:
  ✓ 01_category_revenue_ranking.png - Top 15 categories by revenue
  ✓ 02_margin_vs_revenue_scatter.png - Category efficiency matrix
  ✓ 03_top_bottom_categories_comparison.png - Comparative analysis
  ✓ 04_top_category_trend_forecast.png - Revenue trend & forecast
  ✓ summary_metrics.csv - Key category KPIs

Location: /Users/dominhhien/Documents/AI/Datathon/TuNgayToiGapEm/phan_2_eda/outputs/idea_5/outputs

🔑 KEY FINDINGS:
  - Top category: Streetwear (0.8% revenue)
  - Margin spread: 7.80 pp opportunity
  - Overall margin: 9.68% (benchmark for category strategy)
  - Forecast: top category trending at $-440,085/month

🚀 READY FOR:
  ✓ Category expansion roadmap
  ✓ Margin optimization initiatives
  ✓ Inventory mix tuning
  ✓ Pricing & promotional strategy by category
